# Multi-GPU Inference — Serving Large Models

**Module 27** of the PyTorch Complete Learning Guide

This notebook covers:
- Why multi-GPU inference is necessary for large models
- Model size calculation and GPU requirement estimation
- Tensor Parallel, Pipeline Parallel, and device_map sharding
- KV cache sizing across GPUs
- Quantized inference: FP16 vs INT8 vs INT4
- torch.compile for inference optimization
- Continuous batching concepts
- Latency metrics (TTFT, ITL, throughput)
- Decision tree for choosing a strategy

> All cells run on **CPU only**. Multi-GPU APIs are explained with code patterns.

In [ ]:
import math
import time
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch {torch.__version__}")
HAS_CUDA = torch.cuda.is_available()
print(f"CUDA available: {HAS_CUDA}")
if HAS_CUDA:
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU count: {torch.cuda.device_count()}")

## 1. Why Multi-GPU Inference?

Large language models don't fit on a single GPU. Memory required for **weights only**:

| Model | Params | FP16 Size | INT8 Size | INT4 Size | Min GPUs (A100-80GB, FP16) |
|-------|--------|-----------|-----------|-----------|----------------------------|
| 7B   | 7B     | 14 GB     | 7 GB      | 3.5 GB    | 1 |
| 13B  | 13B    | 26 GB     | 13 GB     | 6.5 GB    | 1 |
| 70B  | 70B    | 140 GB    | 70 GB     | 35 GB     | 2 |
| 405B | 405B   | 810 GB    | 405 GB    | 202 GB    | 11 |

Even models that fit on 1 GPU benefit from multi-GPU for:
- **Lower latency**: Tensor Parallel splits computation across GPUs
- **Higher throughput**: Pipeline Parallel processes multiple requests concurrently

## 2. Model Size Calculator

Let's build a calculator that estimates how many GPUs a model needs.

In [ ]:
@dataclass
class ModelConfig:
    name: str
    vocab_size: int
    hidden_dim: int
    num_layers: int
    num_heads: int
    num_kv_heads: int
    intermediate_dim: int

MODELS = {
    "7B": ModelConfig("Llama-7B", 32000, 4096, 32, 32, 32, 11008),
    "13B": ModelConfig("Llama-13B", 32000, 5120, 40, 40, 40, 13824),
    "70B": ModelConfig("Llama-70B", 32000, 8192, 80, 64, 8, 28672),
    "405B": ModelConfig("Llama-405B", 128000, 16384, 126, 128, 8, 53248),
}

GPU_MEM = {
    "RTX-4090": 24, "A100-40GB": 40, "A100-80GB": 80,
    "H100-80GB": 80, "H200-141GB": 141,
}


def count_params(cfg):
    head_dim = cfg.hidden_dim // cfg.num_heads
    d = cfg.hidden_dim
    per_layer = (
        d * cfg.num_heads * head_dim +       # Q
        d * cfg.num_kv_heads * head_dim +    # K
        d * cfg.num_kv_heads * head_dim +    # V
        cfg.num_heads * head_dim * d +        # output
        d * cfg.intermediate_dim +            # FFN up
        d * cfg.intermediate_dim +            # FFN gate
        cfg.intermediate_dim * d +            # FFN down
        2 * d                                 # norms
    )
    return cfg.vocab_size * d + per_layer * cfg.num_layers + cfg.vocab_size * d + d


for name, cfg in MODELS.items():
    params = count_params(cfg)
    for bits, label in [(16, "FP16"), (8, "INT8"), (4, "INT4")]:
        gb = params * bits / 8 / 1e9
        gpus_a100 = math.ceil(gb / (80 * 0.9))
        print(f"{cfg.name:15s} {label}: {gb:7.1f} GB → {gpus_a100} A100-80GB")

## 3. Strategy 1: Tensor Parallel (TP)

Tensor Parallel splits **each layer's weight matrices** across GPUs. Every GPU participates in every layer.

```
ColwiseParallel (QKV, FFN up):     RowwiseParallel (output, FFN down):
┌──────────────┐                   ┌──────────────┐
│ W [d, 4d]    │                   │ W [4d, d]    │
└──────┬───────┘                   └──────┬───────┘
       │ split columns                    │ split rows
  ┌────┴────┐                        ┌────┴────┐
  │W₀[d,2d] │ W₁[d,2d]│             │W₀[2d,d] │ W₁[2d,d]│
  │  GPU 0  │  GPU 1  │             │  GPU 0  │  GPU 1  │
  └─────────┘─────────┘             └─────────┘─────────┘
  No comm needed                    All-reduce after matmul
```

**Communication**: 2 all-reduce operations per Transformer layer.

In [ ]:
# TP API pattern (requires multi-GPU, shown for reference)
print("""Tensor Parallel API:

from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.tensor.parallel import (
    ColwiseParallel, RowwiseParallel, parallelize_module,
)

mesh = init_device_mesh("cuda", (world_size,), mesh_dim_names=("tp",))

plan = {
    "attn.qkv_proj":  ColwiseParallel(),   # split heads across GPUs
    "attn.out_proj":  RowwiseParallel(),    # all-reduce after
    "ffn.up_proj":    ColwiseParallel(),    # split FFN width
    "ffn.gate_proj":  ColwiseParallel(),
    "ffn.down_proj":  RowwiseParallel(),    # all-reduce after
}

for layer in model.layers:
    parallelize_module(layer, mesh["tp"], plan)
""")

# Communication cost estimation
print("\nTP communication cost per forward pass:")
for model_name, dim, layers in [("7B", 4096, 32), ("70B", 8192, 80)]:
    seq_len = 2048
    comm_bytes = 2 * layers * seq_len * dim * 2  # 2 all-reduces, bf16
    for bw_name, bw in [("NVLink", 900e9), ("PCIe 5", 64e9)]:
        time_ms = comm_bytes / bw * 1000
        print(f"  {model_name} on {bw_name}: {comm_bytes/1e6:.0f} MB, {time_ms:.2f} ms")

## 4. Strategy 2: Pipeline Parallel (PP)

Pipeline Parallel assigns **different layers** to different GPUs:

```
GPU 0: embed + layers 0-15     (receives input)
GPU 1: layers 16-31 + head     (produces output)
```

**Communication**: Only hidden states between stages (point-to-point, not all-reduce).

**Trade-off**: Without micro-batching, only one GPU is active at a time (pipeline bubble).

In [ ]:
def pipeline_bubble(pp_degree, num_microbatches):
    """Calculate pipeline bubble fraction."""
    return (pp_degree - 1) / (pp_degree + num_microbatches - 1)


print("Pipeline bubble analysis:")
print(f"{'PP':>4s}  {'uBatches':>9s}  {'Bubble':>8s}  {'Utilization':>12s}")
print("-" * 40)
for pp in [2, 4, 8]:
    for mb in [4, 8, 16, 32]:
        b = pipeline_bubble(pp, mb)
        print(f"{pp:>4d}  {mb:>9d}  {b:>7.1%}  {1-b:>11.1%}")

## 5. Strategy 3: Simple device_map Sharding

The simplest approach: manually assign model parts to different GPUs.
No distributed library needed, but no within-layer parallelism either.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, 4 * dim, bias=False),
            nn.GELU(),
            nn.Linear(4 * dim, dim, bias=False),
        )

    def forward(self, x):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, need_weights=False)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x


class SimpleLLM(nn.Module):
    def __init__(self, vocab=32000, dim=512, heads=8, layers=8):
        super().__init__()
        self.embed = nn.Embedding(vocab, dim)
        self.layers = nn.ModuleList([TransformerBlock(dim, heads) for _ in range(layers)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)

    def forward(self, input_ids):
        x = self.embed(input_ids)
        for layer in self.layers:
            x = layer(x)
        return self.head(self.norm(x))


# Build on meta device to estimate memory without allocating
with torch.device('meta'):
    model = SimpleLLM(vocab=32000, dim=1024, heads=16, layers=16)

total_params = sum(p.numel() for p in model.parameters())
total_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
print(f"Model: {total_params:,} params ({total_mb:.1f} MB)")


def compute_device_map(model, num_gpus):
    components = []
    for name, module in model.named_children():
        size = sum(p.numel() * p.element_size() for p in module.parameters())
        components.append((name, size))

    total = sum(s for _, s in components)
    per_gpu = total / num_gpus
    device_map, gpu, load = {}, 0, 0
    for name, size in components:
        device_map[name] = f"cuda:{gpu}"
        load += size
        if load >= per_gpu and gpu < num_gpus - 1:
            gpu += 1
            load = 0
    return device_map


for n_gpus in [2, 4]:
    dmap = compute_device_map(model, n_gpus)
    print(f"\nDevice map ({n_gpus} GPUs):")
    for comp, dev in dmap.items():
        m = getattr(model, comp)
        sz = sum(p.numel() * p.element_size() for p in m.parameters()) / 1e6
        print(f"  {comp:10s} → {dev}  ({sz:.1f} MB)")

## 6. KV Cache Sizing Across GPUs

During autoregressive generation, the KV cache stores past keys and values.
With multi-GPU, the cache is distributed differently per strategy:

- **TP**: each GPU caches its head shard (cache splits by heads)
- **PP**: each GPU caches its layer range (cache splits by layers)
- **device_map**: same as PP (layers assigned to devices)

In [ ]:
def kv_cache_size(num_layers, num_kv_heads, head_dim, seq_len,
                  batch_size=1, dtype_bytes=2):
    """KV cache size in bytes: 2 (K+V) * layers * heads * head_dim * seq * batch."""
    return 2 * num_layers * num_kv_heads * head_dim * seq_len * batch_size * dtype_bytes


print("KV Cache Memory (bf16):")
print(f"{'Model':>8s}  {'Batch':>6s}  {'SeqLen':>7s}  {'Total':>8s}  {'TP=2':>8s}  {'TP=4':>8s}  {'TP=8':>8s}")
print("-" * 65)

configs = [
    ("7B",   32, 32, 128),   # layers, kv_heads, head_dim
    ("70B",  80, 8,  128),
    ("405B", 126, 8, 128),
]

for name, layers, kv_heads, hdim in configs:
    for batch in [1, 8, 32]:
        for seq in [2048, 4096]:
            total = kv_cache_size(layers, kv_heads, hdim, seq, batch) / 1e9
            tp2 = total / 2
            tp4 = total / 4
            tp8 = total / 8
            print(f"{name:>8s}  {batch:>6d}  {seq:>7d}  {total:>6.2f}GB  "
                  f"{tp2:>6.2f}GB  {tp4:>6.2f}GB  {tp8:>6.2f}GB")

## 7. Quantized Inference: FP16 vs INT8 vs INT4

Quantization reduces precision to fit larger models on fewer GPUs:

| Precision | Bits/Param | 70B Size | Quality Impact |
|-----------|-----------|----------|----------------|
| FP32      | 32        | 280 GB   | Baseline |
| FP16/BF16 | 16        | 140 GB   | Negligible |
| INT8      | 8         | 70 GB    | < 1% degradation |
| INT4      | 4         | 35 GB    | 1-3% degradation |

In [ ]:
# Compare actual tensor sizes across precisions
dim = 4096
weight_fp32 = torch.randn(dim, dim)
weight_fp16 = weight_fp32.half()
weight_bf16 = weight_fp32.bfloat16()
weight_int8 = weight_fp32.to(torch.int8)

print("Memory for a single [4096, 4096] weight matrix:")
for name, t in [("FP32", weight_fp32), ("FP16", weight_fp16),
                ("BF16", weight_bf16), ("INT8", weight_int8)]:
    mb = t.numel() * t.element_size() / 1e6
    print(f"  {name}: {mb:.1f} MB ({t.element_size()} bytes/element)")

print("\nWhole-model size comparison:")
for model_name in ["7B", "70B", "405B"]:
    cfg = MODELS[model_name]
    params = count_params(cfg)
    print(f"  {cfg.name}:")
    for bits, label in [(32, "FP32"), (16, "FP16"), (8, "INT8"), (4, "INT4")]:
        gb = params * bits / 8 / 1e9
        print(f"    {label}: {gb:6.1f} GB")

In [ ]:
# Quantization API patterns
print("""Quantization APIs:

1. Dynamic quantization (simplest, no calibration):
   quantized = torch.ao.quantization.quantize_dynamic(
       model, {nn.Linear}, dtype=torch.qint8
   )

2. Weight-only quantization (torchao):
   from torchao.quantization import quantize_, int4_weight_only
   quantize_(model, int4_weight_only(group_size=128))

3. Combining with Tensor Parallel:
   # Quantize first, then apply TP
   quantize_(model, int4_weight_only())
   for layer in model.layers:
       parallelize_module(layer, mesh, plan)
""")

## 8. torch.compile for Inference

Three compile modes, each optimized for different inference scenarios:

| Mode | Best For | Trade-off |
|------|----------|----------|
| `default` | General use | Balanced compile time / perf |
| `reduce-overhead` | Latency (CUDA Graphs) | Static shapes required |
| `max-autotune` | Throughput | Longest compile time |

In [ ]:
# CPU demo of torch.compile for inference
model_small = SimpleLLM(vocab=32000, dim=256, heads=4, layers=4)
model_small.eval()
input_ids = torch.randint(0, 32000, (1, 64))

# Eager baseline
with torch.no_grad():
    times = []
    for _ in range(12):
        t0 = time.perf_counter()
        _ = model_small(input_ids)
        times.append(time.perf_counter() - t0)
eager_ms = sum(times[2:]) / len(times[2:]) * 1000
print(f"Eager:    {eager_ms:.2f} ms")

# Compiled
try:
    compiled = torch.compile(model_small, mode="default")
    with torch.no_grad():
        for _ in range(3):  # warmup
            _ = compiled(input_ids)
        times = []
        for _ in range(12):
            t0 = time.perf_counter()
            _ = compiled(input_ids)
            times.append(time.perf_counter() - t0)
    compiled_ms = sum(times[2:]) / len(times[2:]) * 1000
    print(f"Compiled: {compiled_ms:.2f} ms")
    print(f"Speedup:  {eager_ms/compiled_ms:.2f}x")
except Exception as e:
    print(f"Compile not available: {e}")

In [ ]:
# Static shapes for CUDA Graphs compatibility
def pad_to_static(input_ids, pad_id=0, max_len=128):
    """Pad to static length for CUDA Graph / reduce-overhead mode."""
    batch, seq = input_ids.shape
    if seq < max_len:
        padding = torch.full((batch, max_len - seq), pad_id,
                             dtype=input_ids.dtype, device=input_ids.device)
        input_ids = torch.cat([input_ids, padding], dim=1)
    return input_ids


short_input = torch.randint(0, 32000, (1, 37))
padded = pad_to_static(short_input, max_len=128)
print(f"Original shape: {short_input.shape}")
print(f"Padded shape:   {padded.shape}")
print(f"Padding added:  {padded.shape[1] - short_input.shape[1]} tokens")

## 9. AOTInductor for Production

AOTInductor pre-compiles models to `.so` shared libraries for C++ serving:

```
Python Model → torch.export → aot_compile → .so file → C++ Server
```

Benefits:
- No Python GIL (true parallel serving)
- No compilation on first request
- Smaller deployment footprint

In [ ]:
# Demonstrate torch.export (first step of AOTInductor pipeline)
export_model = SimpleLLM(vocab=32000, dim=256, heads=4, layers=4)
export_model.eval()
example = torch.randint(0, 32000, (1, 64))

try:
    ep = torch.export.export(export_model, (example,), strict=False)
    print(f"Export successful!")
    print(f"  Graph nodes:  {len(ep.graph.nodes)}")
    print(f"  Input specs:  {len(ep.graph_signature.input_specs)}")
    print(f"  Output specs: {len(ep.graph_signature.output_specs)}")
    print("\nNext step (requires CUDA):")
    print("  from torch._export import aot_compile")
    print("  so_path = aot_compile(model.cuda(), args=(example.cuda(),))")
except Exception as e:
    print(f"Export demo skipped: {e}")

## 10. Continuous Batching

Static batching wastes GPU cycles when requests have different lengths.
Continuous batching fills empty slots with new requests immediately.

In [ ]:
@dataclass
class Request:
    id: int
    max_tokens: int
    generated: int = 0

    @property
    def done(self):
        return self.generated >= self.max_tokens


class BatchScheduler:
    def __init__(self, max_batch):
        self.max_batch = max_batch
        self.active = {}
        self.waiting = []
        self.done = []
        self.steps = 0

    def add(self, req):
        self.waiting.append(req)

    def step(self):
        self.steps += 1
        # Remove finished
        finished = [rid for rid, r in self.active.items() if r.done]
        for rid in finished:
            self.done.append(self.active.pop(rid))
        # Admit new
        while len(self.active) < self.max_batch and self.waiting:
            r = self.waiting.pop(0)
            self.active[r.id] = r
        # Generate
        for r in self.active.values():
            if not r.done:
                r.generated += 1
        return len(self.active)


sched = BatchScheduler(max_batch=4)
requests = [Request(i, t) for i, t in enumerate([10, 5, 15, 8, 12, 3, 20, 7])]
for r in requests:
    sched.add(r)

print(f"Submitted {len(requests)} requests, max batch={sched.max_batch}")
print(f"{'Step':>5s}  {'Active':>7s}  {'Waiting':>8s}  {'Done':>5s}")
while sched.active or sched.waiting:
    active = sched.step()
    print(f"{sched.steps:>5d}  {active:>7d}  {len(sched.waiting):>8d}  {len(sched.done):>5d}")

total_tokens = sum(r.max_tokens for r in requests)
util = total_tokens / (sched.steps * sched.max_batch)
print(f"\nCompleted in {sched.steps} steps, GPU utilization: {util:.1%}")

## 11. Latency Metrics: TTFT, ITL, Throughput

Three key metrics for inference serving:

```
[Request arrives] → [Process prompt] → [First token] → [Token 2] → ... → [EOS]
                     ├─── TTFT ───────┤
                                       ├── ITL ─┤├── ITL ─┤
```

- **TTFT** (Time to First Token): prefill latency, compute-bound
- **ITL** (Inter-Token Latency): decode latency, memory-bandwidth-bound
- **Throughput**: tokens/sec across all concurrent requests

In [ ]:
# Simulate TTFT and ITL on CPU
model_bench = SimpleLLM(vocab=32000, dim=256, heads=4, layers=4)
model_bench.eval()

# TTFT: time to process entire prompt
prompt = torch.randint(0, 32000, (1, 128))
with torch.no_grad():
    ttft_times = []
    for _ in range(20):
        t0 = time.perf_counter()
        _ = model_bench(prompt)
        ttft_times.append(time.perf_counter() - t0)

ttft_times.sort()
print("TTFT (128-token prompt, CPU):")
print(f"  p50: {ttft_times[9]*1000:.1f} ms")
print(f"  p95: {ttft_times[18]*1000:.1f} ms")
print(f"  p99: {ttft_times[19]*1000:.1f} ms")

# ITL: time per single token decode
single = torch.randint(0, 32000, (1, 1))
with torch.no_grad():
    itl_times = []
    for _ in range(50):
        t0 = time.perf_counter()
        _ = model_bench(single)
        itl_times.append(time.perf_counter() - t0)

itl_times.sort()
print(f"\nITL (single token, CPU):")
print(f"  p50: {itl_times[24]*1000:.2f} ms")
print(f"  p95: {itl_times[47]*1000:.2f} ms")
print(f"  Max tokens/sec: {1.0/itl_times[24]:.0f}")

## 12. Decision Tree: Choosing a Strategy

```
Model fits on 1 GPU?
├─ YES → torch.compile + CUDA Graphs
│        ├─ Latency?    → mode="reduce-overhead"
│        └─ Throughput? → mode="max-autotune" + larger batches
│
└─ NO → How many GPUs?
         ├─ 2-4 GPUs → Tensor Parallel (best latency)
         ├─ 4-8 GPUs → TP + PP hybrid
         └─ Need max throughput? → Pipeline + continuous batching

Additional:
  ├─ Production?         → AOTInductor (.so)
  ├─ Memory constrained? → INT4 quantize first
  └─ Simple setup?       → device_map sharding
```

In [ ]:
def recommend_strategy(params_b, num_gpus, gpu_type="A100-80GB", priority="latency"):
    """Recommend inference strategy based on model and hardware."""
    gpu_mem = GPU_MEM.get(gpu_type, 80)
    usable = gpu_mem * 0.85
    fp16_gb = params_b * 2
    int4_gb = params_b * 0.5

    if fp16_gb <= usable and num_gpus == 1:
        mode = "reduce-overhead" if priority == "latency" else "max-autotune"
        return f"Single GPU + torch.compile(mode='{mode}')"
    elif int4_gb <= usable and num_gpus == 1:
        return "Single GPU + INT4 quantization + torch.compile"
    elif num_gpus <= 4:
        return f"Tensor Parallel (TP={num_gpus})"
    elif num_gpus <= 8:
        return f"Hybrid TP={min(4, num_gpus)} + PP={num_gpus // min(4, num_gpus)}"
    else:
        return f"TP=8 + PP={num_gpus // 8} + continuous batching"


scenarios = [
    (7, 1, "A100-80GB", "latency"),
    (7, 1, "RTX-4090", "throughput"),
    (70, 2, "A100-80GB", "latency"),
    (70, 4, "H100-80GB", "latency"),
    (70, 1, "RTX-4090", "latency"),
    (405, 16, "H100-80GB", "throughput"),
]

print(f"{'Model':>6s}  {'GPUs':>5s}  {'GPU Type':>12s}  {'Priority':>10s}  Strategy")
print("-" * 80)
for params, gpus, gpu, prio in scenarios:
    strat = recommend_strategy(params, gpus, gpu, prio)
    print(f"{params:>5d}B  {gpus:>5d}  {gpu:>12s}  {prio:>10s}  {strat}")

## 13. Exercise: Minimum GPUs for 70B INT4

**Task**: Calculate the minimum number of GPUs needed to serve a 70B parameter model
with INT4 quantization on different GPU types. Include KV cache for batch_size=16, seq_len=4096.

In [ ]:
# Exercise: fill in the calculations
model_params = 70e9
int4_weight_gb = model_params * 4 / 8 / 1e9  # INT4 = 4 bits per param

# KV cache: 70B has 80 layers, 8 KV heads, head_dim=128
kv_gb = kv_cache_size(80, 8, 128, seq_len=4096, batch_size=16, dtype_bytes=2) / 1e9

total_gb = int4_weight_gb + kv_gb

print(f"70B model INT4 weights: {int4_weight_gb:.1f} GB")
print(f"KV cache (batch=16, seq=4096, bf16): {kv_gb:.1f} GB")
print(f"Total: {total_gb:.1f} GB")
print()

for gpu_name, mem in GPU_MEM.items():
    usable = mem * 0.85
    gpus_needed = math.ceil(total_gb / usable)
    fits_one = "YES" if total_gb <= usable else "NO"
    print(f"  {gpu_name:15s} ({mem:3d} GB): fits on 1? {fits_one:3s}  "
          f"min GPUs: {gpus_needed}")

## Key Takeaways

1. **Model size determines strategy**: 70B+ models in FP16 need multiple GPUs. INT4 can reduce GPU count by 4x.

2. **Three main strategies**:
   - **Tensor Parallel**: split layers across GPUs, best latency, needs NVLink
   - **Pipeline Parallel**: stack layers on different GPUs, best throughput with micro-batching
   - **device_map**: simplest, no distributed library, but sequential execution

3. **KV cache is significant**: at batch_size=32, the KV cache alone can be 40+ GB for 70B models.

4. **Quantize first**: INT4/INT8 often reduces GPU requirements more than adding GPUs.

5. **torch.compile matters**: `reduce-overhead` (CUDA Graphs) for latency, `max-autotune` for throughput.

6. **Continuous batching** improves throughput by 30-50% over static batching.

7. **AOTInductor** eliminates Python overhead for production C++ serving.

8. **Measure TTFT, ITL, and throughput** — optimizing one can hurt others.

---

**Next**: This is the latest module. Check the [README](../README.md) for updates.

**Previous**: [Module 26 — Memory Profiling & Optimization](../26_memory_profiling/)